# Usage

In the example below the reading of the Safecast LOG format is
presented. Similarly, the ERS format can be read using:

In [1]:
!pip install ../.. gdal==3.8.4 --upgrade
from radiation_toolbox_reader.ers import ERSReader as Reader

Processing /home/martin/git/opengeolabs/radiation-toolbox/radiation-toolbox-reader
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for radiation-toolbox-reader: filename=radiation_toolbox_reader-1.0.0-py3-none-any.whl size=29629 sha256=9d70d19d70325ce5a9b143794c61194933774451848ef6cf3e34f6f226099b9b
  Stored in directory: /home/martin/.cache/pip/wheels/59/49/6e/da3a3b2a2296d8284881938ce572395217fa12af8296fe9966
Successfully built radiation-toolbox-reader
  Attempting uninstall: radiation-toolbox-reader
    Found existing installation: radiation-toolbox-reader 1.0.0
    Uninstalling radiation-toolbox-reader-1.0.0:
      Successfully uninstalled radiation-toolbox-reader-1.0.0


First import a reader class:

In [2]:
from radiation_toolbox_reader.safecast import SafecastReader as Reader

Input data file is read by:

In [3]:
input_file = "../../tests/data/sample.log"
r = Reader(input_file)

Basic metadata is provided by `metadata` property.

In [4]:
r.metadata

{'table': 'safecast_metadata',
 'columns': {'filename': 'sample.log',
  'format': '1.3.4nano',
  'deadtime': 'on',
  'callibration_coefficient': 0.0029940119760479}}

Number of records is returned by the ``count()`` method:

In [5]:
r.count()

941

Overall statistics may be obtained by the `stats()` method.

*Tip:* Or using Context Manager Protocol:

In [6]:
with Reader(input_file) as r:
    print(r.count())

941


Attribute definitions (name, data type, metadata) may be return by the
``attributeDefs()`` method:

In [7]:
attrs = r.attributeDefs()

for name in list(attrs.keys())[:3]:
    print(name, attrs[name])

device {'type': <class 'str'>, 'alias': 'Device', 'computed': 0}
device_id {'type': <class 'str'>, 'alias': 'Device ID', 'computed': 0}
date_time {'type': <class 'str'>, 'alias': 'Datetime', 'computed': 0}


Let's print first record:

In [8]:
first = next(r)
first

SafecastRecord([('device', '$BNRDD'),
                ('device_id', '2849'),
                ('date_time', '2019-01-01T13:00:36Z'),
                ('cpm', 31),
                ('pulses5s', 2),
                ('pulses_total', 1049),
                ('validity', 'A'),
                ('lat_deg', '5038.5187'),
                ('hemisphere', 'N'),
                ('long_deg', '01351.3902'),
                ('east_west', 'E'),
                ('altitude', 256.2),
                ('gps_validity', 'A'),
                ('sat', 6),
                ('hdop', 151),
                ('checksum', '*48')])

Record coordinates (longitude, latitude) may be obtained by `point` property.

In [9]:
first.point

(13.856503333333333, 50.641978333333334)

## Export

Data processed by the reader may to exported into:

- plain CSV by ``exportCSV()`` method, or
- geospatial formats by ``export()`` method

Currently, the export method supports two geospatial data formats:

- OGC GeoPackage
- SQLite

In [10]:
output_gpkg = "/tmp/sample.gpkg"
r.export(output_gpkg, "GPKG", overwrite=True)

By default for each imported LOG file a new table in target GDAL data source is created.

In [11]:
with Reader("../../tests/data/sample_1.log") as r:
    r.export(output_gpkg, "GPKG")

In our case, the output GDAL datasource contains two layers corresponding to the input LOG files:

In [12]:
from osgeo import gdal

def print_db_layers(gpkg):
    ds = gdal.OpenEx(gpkg, gdal.OF_VECTOR)
    for idx in range(ds.GetLayerCount()):
        print(ds.GetLayer(idx).GetName())

print_db_layers(output_gpkg)

sample
sample_1
safecast_metadata


By `single_table=data` new data records are be inserted into single table named "data".

In [13]:
output_gpkg_1 = "/tmp/sample_1.gpkg"
with Reader("../../tests/data/sample.log") as r:
    r.export(output_gpkg_1, "GPKG", overwrite=True, single_table="data")
with Reader("../../tests/data/sample_1.log") as r:
    r.export(output_gpkg_1, "GPKG", single_table="data")

print_db_layers(output_gpkg_1)

data
safecast_metadata


## Safecast extension

In addition to reading input data, ``SafecastReader`` can calculate some additional attributes:

In [14]:
with Reader(input_file, computed_attributes=True) as r:
    print(next(r))

SafecastRecord([('device', '$BNRDD'), ('device_id', '2849'), ('date_time', '2019-01-01T13:00:36Z'), ('cpm', 31), ('pulses5s', 2), ('pulses_total', 1049), ('validity', 'A'), ('lat_deg', '5038.5187'), ('hemisphere', 'N'), ('long_deg', '01351.3902'), ('east_west', 'E'), ('altitude', 256.2), ('gps_validity', 'A'), ('sat', 6), ('hdop', 151), ('checksum', '*48'), ('ader_microsvh', 0.0718562874251496), ('time_local', '14:00:36'), ('speed_kmph', 0), ('dose_increment', 0), ('time_cumulative', '00:00:00'), ('dose_cumulative', 0), ('dist_cumulative', 0)])


In the case of computed attributes also overall statistics may be returned by `stats()`.

In [15]:
r.stats()

{'count': 941,
 'radiation': {'max': 2.2634730538922123,
  'avg': 0.2691397226800388,
  'total': 0.35262890884896525},
 'route': {'speed': 2.3288569981172658,
  'time': '01:18:34',
  'distance': 3051.8665013830273}}